# M5.1 · Laboratorio de Git y GitHub

**Modalidad:** presencial · **Encuentro:** 180 minutos · **Trabajo en notebook:** 80–90 minutos.

## Propósito

Construir un recorrido de cambios que otra persona pueda comprender y reproducir: rama, al menos dos commits descriptivos, exclusión de archivos sensibles y Pull Request.

## Resultados observables

- Distinguir qué resuelve Git y qué aporta GitHub.
- Predecir y comprobar cambios de estado con `status` y `diff`.
- Organizar una mejora en una rama y al menos dos commits con propósito único.
- Evitar que archivos sensibles ingresen al historial.
- Preparar la evidencia necesaria para abrir un Pull Request.

**Evidencia mínima:** historial local legible, archivo sensible ignorado y URL o captura verificable del PR. La práctica se realiza individualmente o en equipos estables, sin rotación.


## 1. Antes de ejecutar: Git no es GitHub

En una **nueva celda de texto debajo de esta consigna**, completá una frase para cada comienzo:

- **Git** conserva…
- **GitHub** permite…

Después respondé en la misma celda: **¿qué evidencia aporta un PR que no aporta enviar un archivo final?**

No ejecutes todavía las celdas de Git: primero dejá escrita tu respuesta.


## 2. Ejemplo trabajado · iniciar y comprobar

Ejecutá las **dos celdas de código siguientes, en orden y una sola vez**. La primera crea un repositorio temporal y la segunda muestra el último commit y los archivos versionados. No tenés que modificar ese código.

El repositorio es **temporal y local**: no usa credenciales ni publica nada. Desaparece al cerrar o reiniciar el kernel; antes de terminar, conservá la evidencia en el repositorio del curso o exportá el plan B local.


In [1]:
from pathlib import Path
import shutil
import subprocess
import tempfile

assert shutil.which("git"), "Git no está disponible en este entorno."

entorno_temporal = tempfile.TemporaryDirectory(prefix="m5_git_")
repo = Path(entorno_temporal.name)

def git(*args, check=True):
    resultado = subprocess.run(
        ["git", "-C", str(repo), *args],
        text=True,
        capture_output=True,
        check=check,
    )
    return (resultado.stdout or resultado.stderr).strip()

git("init", "-q")
git("config", "user.name", "Práctica M5")
git("config", "user.email", "practica@example.invalid")
(repo / "README.md").write_text("# Análisis reproducible\n", encoding="utf-8")
git("add", "README.md")
git("commit", "-q", "-m", "inicia repositorio reproducible")

print("Repositorio local de práctica:", repo)
print(git("status", "--short", "--branch"))


Repositorio local de práctica: /var/folders/28/d3klhtz53w3909ps3gy013wc0000gp/T/m5_git_k5628pds
## master


In [2]:
print("Último commit:")
print(git("log", "-1", "--oneline"))
print("\nArchivos versionados:")
print(git("ls-files"))


Último commit:
5081258 inicia repositorio reproducible

Archivos versionados:
README.md


## 3. Práctica guiada · predecir, ejecutar, comprobar

**Propósito:** reconocer el efecto de una rama y un cambio antes de confirmarlo.  
**Procedimiento:** elegí un nombre de rama, anticipá qué mostrará `status`, ejecutá y contrastá.  
**Soporte:** repositorio temporal, funciones `git()` y pistas graduadas.  
**Producto:** rama creada y explicación breve de la diferencia entre archivo modificado, preparado y confirmado.  
**Criterio:** la explicación usa evidencia de `status` o `diff`, no sólo memoria.

### Qué hacer

1. Verificá que ya ejecutaste las dos celdas del punto 2.
2. Antes de tocar el código, escribí en una nueva celda de texto qué esperás que muestre `git status` después de modificar `README.md`.
3. En la celda siguiente reemplazá únicamente los dos valores `None`: `nombre_rama` por un nombre breve y `mensaje_readme` por un mensaje que explique el cambio.
4. Ejecutá la celda **una sola vez**. El código crea la rama, modifica el README, muestra `status` y `diff`, prepara el archivo y genera el commit.
5. Debajo del resultado, agregá una celda de texto de 3 a 5 líneas que explique cuándo el archivo estuvo **modificado**, **preparado** y **confirmado**, citando una salida observada.

Si ejecutás la celda por segunda vez, la rama ya existirá. En ese caso reiniciá el kernel y comenzá desde el punto 2, o elegí otro nombre de rama.


In [4]:
# TODO: reemplazá los None por un nombre breve y un mensaje descriptivo.
nombre_rama = "Ajuste"
mensaje_readme = "Prueba de commit con README.md actualizado"

if nombre_rama is None or not mensaje_readme:
    print("Pendiente: definí nombre_rama y mensaje_readme; luego volvé a ejecutar.")
else:
    git("switch", "-c", nombre_rama)
    (repo / "README.md").write_text(
        "# Análisis reproducible\n\n## Ejecución\nDescribir aquí los pasos.\n",
        encoding="utf-8",
    )
    print(git("status", "--short"))
    print(git("diff", "--", "README.md"))
    git("add", "README.md")
    print("Cambio preparado:\n", git("diff", "--staged", "--stat"))
    git("commit", "-q", "-m", mensaje_readme)
    assert git("status", "--short") == ""
    print("Confirmado:", git("log", "-1", "--oneline"))


M README.md
diff --git a/README.md b/README.md
index 4814338..04d5bf9 100644
--- a/README.md
+++ b/README.md
@@ -1 +1,4 @@
 # Análisis reproducible
+
+## Ejecución
+Describir aquí los pasos.
Cambio preparado:
 README.md | 3 +++
 1 file changed, 3 insertions(+)
Confirmado: fabfc95 Prueba de commit con README.md actualizado


### Ayudas graduadas

1. `git status --short` distingue cambios sin preparar y preparados.
2. `git diff` muestra cambios sin preparar; `git diff --staged`, los ya agregados.
3. Un commit debe responder: **¿qué decisión registra?**


## 4. Desafío principal · historial y protección de secretos

**Propósito:** producir un historial pequeño que pueda revisarse.  
**Procedimiento:** realizá al menos dos mejoras separadas, inspeccioná cada diff, agregá sólo lo necesario y confirmá con mensajes descriptivos. Creá un archivo `.env.demo`, agregalo a `.gitignore` y verificá que Git lo ignore.  
**Soporte:** comandos de la presentación, `status`, `diff`, `log` y `check-ignore`.  
**Producto:** al menos dos commits de trabajo en una rama, `.gitignore` y comprobaciones visibles.  
**Criterios:** un propósito por commit; ningún secreto versionado; mensajes comprensibles sin abrir el código.

No uses credenciales reales: el archivo de demostración debe contener únicamente `TOKEN=valor_de_ejemplo`.

### Qué hacer

1. En la celda siguiente reemplazá únicamente `mensaje_calculo` y `mensaje_ignorado` por dos mensajes de commit diferentes y descriptivos.
2. Ejecutá la celda una sola vez. El código crea `analisis.py`, `.env.demo` y `.gitignore`, y realiza las comprobaciones.
3. Confirmá que la salida termine con `Ignorado: .env.demo` y `Estado final: limpio`.
4. Agregá una celda de texto con dos respuestas: por qué los cambios se separaron en commits distintos y por qué `.env.demo` no debe aparecer entre los archivos versionados.


In [ ]:
# TODO: reemplazá None por mensajes descriptivos antes de ejecutar.
mensaje_calculo = None
mensaje_ignorado = None

if not mensaje_calculo or not mensaje_ignorado:
    print("Pendiente: definí los dos mensajes y ejecutá nuevamente.")
else:
    (repo / "analisis.py").write_text(
        "def total(valores):\n    return sum(valores)\n",
        encoding="utf-8",
    )
    print("Cambio 1 antes de confirmar:\n", git("status", "--short"))
    git("add", "analisis.py")
    git("commit", "-q", "-m", mensaje_calculo)

    (repo / ".env.demo").write_text("TOKEN=valor_de_ejemplo\n", encoding="utf-8")
    (repo / ".gitignore").write_text(".env*\n!.env.example\n", encoding="utf-8")
    print("Cambio 2 antes de confirmar:\n", git("status", "--short"))
    git("add", ".gitignore")
    git("commit", "-q", "-m", mensaje_ignorado)

    rama_actual = git("branch", "--show-current")
    historial = git("log", "--oneline", "--decorate").splitlines()
    ignorado = git("check-ignore", ".env.demo")
    versionados = git("ls-files").splitlines()

    assert rama_actual not in {"main", "master"}, "Trabajá en una rama nueva."
    assert len(historial) >= 4, "Faltan los commits de la práctica guiada o del desafío."
    assert ignorado == ".env.demo", "Git todavía no ignora .env.demo."
    assert ".env.demo" not in versionados, "El archivo sensible quedó versionado."
    assert git("status", "--short") == "", "El repositorio conserva cambios sin confirmar."
    print("Rama:", rama_actual)
    print("Historial:\n" + "\n".join(historial))
    print("Ignorado:", ignorado)
    print("Estado final:", git("status", "--short") or "limpio")


## 5. Transferencia a GitHub

Esta parte se realiza **fuera del repositorio temporal del notebook**:

1. Abrí el repositorio del curso en tu equipo.
2. Repetí allí una mejora pequeña en una rama propia, con al menos dos commits descriptivos.
3. Publicá esa rama en GitHub y abrí un Pull Request.
4. Pegá debajo, en una celda de texto, la URL del PR o indicá dónde guardaste la captura.

El notebook no solicita ni almacena credenciales.

Checklist:

- [ ] El PR explica el problema y la solución.
- [ ] El diff contiene sólo archivos necesarios.
- [ ] Al menos dos commits se comprenden sin abrir el código.
- [ ] No hay credenciales, datos privados ni archivos pesados.
- [ ] El README permite reproducir el resultado.

**Plan B:** si GitHub no está disponible, entregá `git log --oneline --decorate`, `git status --short` y el diff local.


## 6. Ticket de salida

En una nueva celda de texto, escribí dos frases: una decisión que cambió después de revisar `status` o `diff`, y una regla que aplicarás en el proyecto integrador.
